# GNN MAPPO (based on Li. et al)

### Guarded Territory environment — custom VMAS scenario for GNN communication ablations

**Changes from previous version:**
- Fixed: `intercpetors` typo → `interceptors` throughout
- Fixed: `in None` → `is None` in reward()
- Fixed: missing interceptor loop in team bonus calculation
- Fixed: removed broken `_intruder_action_script` (process_action handles it)
- Fixed: removed dead `self.actionHead` from CommPolicy
- Fixed: `torch.long` → `torch.float32` for continuous actions in buffer
- Rewrote: GNNTrainer to use GuardedTerritoryAdapter instead of PettingZoo
- Rewrote: Experiment sweep for VMAS (action_dim=2, no GraphBuilder)

### Setup & Dependencies

In [32]:
%pip -q install vmas matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import os
import vmas

from torch.optim import Adam
from torch.distributions import Normal

### Guarded Territory Scenario

Custom VMAS scenario with heterogeneous agents (scouts + interceptors) vs scripted intruders.

**Fixes applied:**
- `self.interceptors` spelling consistent throughout
- `is None` instead of `in None`
- Team bonus loop now iterates over interceptors
- Removed broken `action_script`; intruders controlled solely via `process_action`

In [ ]:
import typing
from vmas.simulator.core import Agent, World, Landmark, Sphere
from vmas.simulator.scenario import BaseScenario
from vmas.simulator.utils import Color, ScenarioUtils

# Agent Type definitions
SCOUT = "scout"
INTERCEPTOR = "interceptor"
INTRUDER = "intruder"


class Scenario(BaseScenario):
    """
    Guarded Territory: heterogeneous cooperative-competitive MARL scenario.

    Designed for GNN communication ablations with:
    - Heterogeneous observation spaces (scouts see further, interceptors see closer)
    - Communication constraints (GNN is the only inter-agent channel)
    - Mixed cooperative-competitive dynamics (defenders cooperate, intruders compete)

    Key Principle: Interceptors CANNOT succeed without scout communication.
    Ensures that r_comm and K ablations produce strong, interpretable signals.
    """

    def make_world(self, batch_dim: int, device: torch.device, **kwargs):
        # Agent / World Config
        self.n_scouts = kwargs.get("n_scouts", 3)
        self.n_interceptors = kwargs.get("n_interceptors", 3)
        self.n_intruders = kwargs.get("n_intruders", 3)
        self.n_zones = kwargs.get("n_zones", 2)
        self.world_size = kwargs.get("world_size", 5.0)

        # Observation Radii - core asymmetry
        self.scout_fov = kwargs.get("scout_fov", 1.0)
        self.interceptor_fov = kwargs.get("interceptor_fov", 0.5)
        self.tag_radius = kwargs.get("tag_radius", 0.1)

        # Speeds
        self.intruder_speed = kwargs.get("intruder_speed", 0.5)
        self.defender_speed = kwargs.get("defender_speed", 0.8)

        self.n_defenders = self.n_scouts + self.n_interceptors

        self.intruder_skill = kwargs.get("intruder_skill", 0.0)

        # Create World
        world = World(
            batch_dim=batch_dim,
            device=device,
            dt=0.1,
            drag=0.25,
            dim_c=0,  # no built-in comms — GNN handles this
            x_semidim=self.world_size,
            y_semidim=self.world_size,
        )

        # Create Scouts
        self.scouts = []
        for i in range(self.n_scouts):
            agent = Agent(
                name=f"scout_{i}",
                collide=True,
                mass=1.0,
                shape=Sphere(radius=0.075),
                max_speed=self.defender_speed,
                color=Color.BLUE,
                u_range=1.0,
            )
            agent.agent_type = SCOUT
            agent.type_id = 0
            world.add_agent(agent=agent)
            self.scouts.append(agent)

        # Create Interceptors  [FIX: consistent spelling]
        self.interceptors = []
        for i in range(self.n_interceptors):
            agent = Agent(
                name=f"interceptor_{i}",
                collide=True,
                mass=1.0,
                shape=Sphere(radius=0.09),
                max_speed=self.defender_speed,
                color=Color.GREEN,
                u_range=1.0,
            )
            agent.agent_type = INTERCEPTOR
            agent.type_id = 1
            world.add_agent(agent)
            self.interceptors.append(agent)

        # Create Intruders (scripted via process_action)
        # [FIX: removed action_script parameter — process_action handles everything]
        self.intruders = []
        for i in range(self.n_intruders):
            intruder = Agent(
                name=f"intruder_{i}",
                collide=True,
                mass=1.0,
                shape=Sphere(radius=0.075),
                max_speed=self.intruder_speed,
                color=Color.RED,
                u_range=1.0,
            )
            intruder.agent_type = INTRUDER
            intruder.type_id = 2
            world.add_agent(intruder)
            self.intruders.append(intruder)

        self.defenders = self.scouts + self.interceptors

        # Create Target Zones / Landmarks
        self.zones = []
        for i in range(self.n_zones):
            zone = Landmark(
                name=f"zone_{i}",
                collide=False,
                movable=False,
                shape=Sphere(radius=0.2),
                color=Color.LIGHT_GREEN,
            )
            world.add_landmark(zone)
            self.zones.append(zone)

        # Tracking Tensors (allocated in reset)
        self._intruder_tagged = None
        self._zone_breached = None
        self._tag_count = None

        return world

    def reset_world_at(self, env_index: typing.Optional[int] = None):
        batch = self.world.batch_dim
        device = self.world.device
        self._intruder_speed_var = self.intruder_speed * (0.8 + 0.4 * torch.rand(1).item())

        if env_index is None:
            self._intruder_tagged = torch.zeros(
                batch, self.n_intruders, dtype=torch.bool, device=device
            )
            self._zone_breached = torch.zeros(
                batch, self.n_zones, dtype=torch.bool, device=device
            )
            self._tag_count = torch.zeros(batch, dtype=torch.float32, device=device)
            self._prev_intruder_tagged = self._intruder_tagged.clone()
            self._prev_zone_breached = self._zone_breached.clone()
        else:
            self._intruder_tagged[env_index] = False
            self._zone_breached[env_index] = False
            self._tag_count[env_index] = 0.0
            self._prev_intruder_tagged[env_index] = False
            self._prev_zone_breached[env_index] = False

        if env_index is None:
            self._intruder_targets = torch.randint(
                0, self.n_zones, (batch, self.n_intruders), device=device
            )
        else:
            self._intruder_targets[env_index] = torch.randint(
                0, self.n_zones, (self.n_intruders,), device=device
            )
        # Spawn zones in inner region
        for i, zone in enumerate(self.zones):
            pos = torch.zeros(
                (1, 2) if env_index is not None else (batch, 2),
                dtype=torch.float32, device=device,
            )
            angle = 2 * torch.pi * i / self.n_zones
            radius = 0.3
            pos[..., 0] = radius * torch.cos(torch.tensor(angle))
            pos[..., 1] = radius * torch.sin(torch.tensor(angle))
            pos += 0.1 * torch.randn_like(pos)
            zone.set_pos(pos, batch_index=env_index)

        # Spawn defenders near zones (inner ring)
        for i, defender in enumerate(self.defenders):
            pos = torch.zeros(
                (1, 2) if env_index is not None else (batch, 2),
                dtype=torch.float32, device=device,
            )
            angle = 2 * torch.pi * i / self.n_defenders
            radius = 0.5 + 0.2 * torch.rand(pos.shape[0], 1, device=device)
            pos[..., 0:1] = radius * torch.cos(torch.tensor(angle))
            pos[..., 1:2] = radius * torch.sin(torch.tensor(angle))
            pos += 0.05 * torch.randn_like(pos)
            defender.set_pos(pos, batch_index=env_index)

        # Spawn intruders on outer edge
        for i, intruder in enumerate(self.intruders):
            pos = torch.zeros(
                (1, 2) if env_index is not None else (batch, 2),
                dtype=torch.float32, device=device,
            )
            angle = 2 * torch.pi * torch.rand(1, device=device).item()
            radius = self.world_size * (0.7 + 0.2 * torch.rand(1, device=device).item())
            pos[..., 0] = radius * torch.cos(torch.tensor(angle))
            pos[..., 1] = radius * torch.sin(torch.tensor(angle))
            intruder.set_pos(pos, batch_index=env_index)

    def _get_intruder_actions(self, intruder: Agent) -> torch.Tensor:
      if not hasattr(self, '_intruder_speed_var'):
          self._intruder_speed_var = self.intruder_speed

      d, b = self.world.device, self.world.batch_dim

      # ── Skilled component (existing goal-seeking + evasion) ──
      intruder_idx = self.intruders.index(intruder)
      target_zone = self.zones[intruder_idx % self.n_zones]
      to_target = target_zone.state.pos - intruder.state.pos
      dist_to_target = torch.linalg.vector_norm(to_target, dim=-1, keepdim=True) + 1e-6
      dir_target = to_target / dist_to_target

      evasion_radius = 0.6
      evade_dir = torch.zeros(b, 2, device=d)
      min_ic_dist = torch.full((b, 1), float("inf"), device=d)

      for interceptor in self.interceptors:
          away = intruder.state.pos - interceptor.state.pos
          ic_dist = torch.linalg.vector_norm(away, dim=-1, keepdim=True) + 1e-6
          closer = ic_dist < min_ic_dist
          min_ic_dist = torch.where(closer, ic_dist, min_ic_dist)
          evade_dir = torch.where(closer.expand_as(away), away / ic_dist, evade_dir)

      evade_weight = torch.clamp(1.0 - min_ic_dist / evasion_radius, min=0.0, max=0.8)
      goal_weight = 1.0 - evade_weight

      skilled_dir = goal_weight * dir_target + evade_weight * evade_dir
      skilled_dir = skilled_dir / (torch.linalg.vector_norm(skilled_dir, dim=-1, keepdim=True) + 1e-6)

      # ── Random component ──
      random_dir = torch.randn(b, 2, device=d)
      random_dir = random_dir / (torch.linalg.vector_norm(random_dir, dim=-1, keepdim=True) + 1e-6)

      # ── Blend by skill level ──
      skill = self.intruder_skill
      direction = skill * skilled_dir + (1 - skill) * random_dir
      direction = direction / (torch.linalg.vector_norm(direction, dim=-1, keepdim=True) + 1e-6)

      noise = 0.15 * torch.randn(b, 2, device=d)
      action = self._intruder_speed_var * (direction + noise)
      return action

    def process_action(self, agent: Agent):
        """Override to inject scripted actions for intruders."""
        if hasattr(agent, "agent_type") and agent.agent_type == INTRUDER:
            agent.action.u = self._get_intruder_actions(agent)

    def observation(self, agent: Agent) -> torch.Tensor:
        batch = self.world.batch_dim
        device = self.world.device

        if hasattr(agent, "agent_type") and agent.agent_type == SCOUT:
            fov = self.scout_fov
            type_oh = torch.tensor([1.0, 0.0], device=device).expand(batch, 2)
        elif hasattr(agent, "agent_type") and agent.agent_type == INTERCEPTOR:
            fov = self.interceptor_fov
            type_oh = torch.tensor([0.0, 1.0], device=device).expand(batch, 2)
        else:
            return torch.zeros(batch, 2, device=device)

        obs_parts = []
        obs_parts.append(agent.state.vel)
        obs_parts.append(agent.state.pos)
        obs_parts.append(type_oh)

        for zone in self.zones:
            obs_parts.append(zone.state.pos - agent.state.pos)

        for intruder in self.intruders:
            rel_pos = intruder.state.pos - agent.state.pos
            dist = torch.linalg.vector_norm(rel_pos, dim=-1, keepdim=True)
            visible = (dist <= fov).float()
            obs_parts.append(rel_pos * visible)
            obs_parts.append(intruder.state.vel * visible)

        for other in self.defenders:
            if other is agent:
                continue
            rel_pos = other.state.pos - agent.state.pos
            dist = torch.linalg.vector_norm(rel_pos, dim=-1, keepdim=True)
            visible = (dist <= fov).float()
            obs_parts.append(rel_pos * visible)

        return torch.cat(obs_parts, dim=-1)

    def pre_step(self):
        if self._intruder_tagged is None:
            return

        self._prev_intruder_tagged = self._intruder_tagged.clone()
        self._prev_zone_breached = self._zone_breached.clone()

        for j, intruder in enumerate(self.intruders):
            already_tagged = self._intruder_tagged[:, j]

            for interceptor in self.interceptors:  # [FIX: consistent spelling]
                dist = torch.linalg.vector_norm(
                    interceptor.state.pos - intruder.state.pos, dim=-1
                )
                just_tagged = (~already_tagged) & (dist < self.tag_radius)
                self._intruder_tagged[:, j] = self._intruder_tagged[:, j] | just_tagged
                self._tag_count += just_tagged.float()
        # Compute zone breach events
        for k, zone in enumerate(self.zones):
            for j, intruder in enumerate(self.intruders):
                tagged = self._intruder_tagged[:, j]
                dist_to_zone = torch.linalg.vector_norm(
                    intruder.state.pos - zone.state.pos, dim=-1
                )
                breached = (~tagged) & (dist_to_zone < 0.15)
                new_breached = breached & (~self._zone_breached[:, k])
                self._zone_breached[:, k] = self._zone_breached[:, k] | breached


    def reward(self, agent: Agent) -> torch.Tensor:
        if hasattr(agent, "agent_type") and agent.agent_type == INTRUDER:
            return torch.zeros(self.world.batch_dim, device=self.world.device)

        batch = self.world.batch_dim
        device = self.world.device
        rew = torch.zeros(batch, device=device)

        new_tags = self._intruder_tagged & (~self._prev_intruder_tagged)
        n_new_tags = new_tags.float().sum(dim=-1)
        rew += 3.0 * n_new_tags

        new_breaches = self._zone_breached & (~self._prev_zone_breached)
        n_new_breaches = new_breaches.float().sum(dim=-1)
        rew -= 5.0 * n_new_breaches

        # Individual shaping
        if hasattr(agent, "agent_type") and agent.agent_type == SCOUT:
            for intruder in self.intruders:
                dist_to_intruder = torch.linalg.vector_norm(
                    agent.state.pos - intruder.state.pos, dim=-1
                )
                sees_intruder = (dist_to_intruder < self.scout_fov).float()
                rew += 0.5 * sees_intruder

                for interceptor in self.interceptors:
                    dist_to_interceptor = torch.linalg.vector_norm(
                        agent.state.pos - interceptor.state.pos, dim=-1
                    )
                    in_relay_range = ((dist_to_interceptor > 0.2) & (dist_to_interceptor < 1.0)).float()
                    rew += 0.2 * sees_intruder * in_relay_range

        elif hasattr(agent, "agent_type") and agent.agent_type == INTERCEPTOR:
            min_dist = torch.full((batch,), float("inf"), device=device)
            best_closing_speed = torch.zeros(batch, device=device)
            for j, intruder in enumerate(self.intruders):
                if self._intruder_tagged is not None:
                    tagged = self._intruder_tagged[:, j]
                else:
                    tagged = torch.zeros(batch, dtype=torch.bool, device=device)
                
                rel_pos = intruder.state.pos - agent.state.pos
                dist = torch.linalg.vector_norm(rel_pos, dim=-1)
                # closing speed
                rel_vel = agent.state.vel - intruder.state.vel
                closing = -(rel_vel * rel_pos).sum(dim=-1) / (dist + 1e-6)

                effective_dist = torch.where(
                    tagged, torch.tensor(float('inf'), device=device), dist
                )
                closer_mask = effective_dist < min_dist
                min_dist = torch.where(closer_mask, effective_dist, min_dist)
                best_closing_speed = torch.where(closer_mask, closing, best_closing_speed)

            min_dist = torch.clamp(min_dist, max=5.0)
            rew -= 0.5 * min_dist # penalize dist
            rew += 0.3 * best_closing_speed.clamp(-1, 1)

            # proximity bonus
            near_tag = torch.clamp(1.0 - min_dist / (self.tag_radius * 3), min=0)
            rew += 2.0 * near_tag
        return rew

    def done(self) -> torch.Tensor:
        if self._intruder_tagged is None or self._zone_breached is None:
            return torch.zeros(
                self.world.batch_dim, dtype=torch.bool, device=self.world.device
            )
        all_tagged = self._intruder_tagged.all(dim=-1)
        all_breached = self._zone_breached.all(dim=-1)
        return all_tagged | all_breached

    def info(self, agent: Agent) -> dict:
        info = {}
        if self._intruder_tagged is not None:
            info["n_tagged"] = self._intruder_tagged.sum(dim=-1).float()
        if self._zone_breached is not None:
            info["n_breached"] = self._zone_breached.sum(dim=-1).float()
        return info


def get_obs_dim(n_scouts=3, n_interceptors=3, n_intruders=3, n_zones=2):
    n_defenders = n_scouts + n_interceptors
    return (
        2 + 2 + 2                  # vel + pos + type_one_hot
        + n_zones * 2              # zone relative positions
        + n_intruders * 2          # intruder rel pos (masked)
        + n_intruders * 2          # intruder vel (masked)
        + (n_defenders - 1) * 2    # other defenders rel pos
    )

### VMAS Adapter

Bridges the VMAS scenario to our GNN-MAPPO training loop.
Filters out scripted intruders, stacks defender tensors, provides adj matrix construction.

In [34]:
class GuardedTerritoryAdapter:
    def __init__(
        self,
        num_envs: int = 1,
        device: str = "cpu",
        n_scouts: int = 3,
        n_interceptors: int = 3,
        n_intruders: int = 3,
        n_zones: int = 2,
        max_steps: int = 200,
        **kwargs,
    ):
        self.num_envs = num_envs
        self.device = device
        self.n_scouts = n_scouts
        self.n_interceptors = n_interceptors
        self.n_intruders = n_intruders
        self.n_defenders = n_scouts + n_interceptors
        self.n_zones = n_zones

        self.env = vmas.make_env(
            scenario=Scenario(),
            num_envs=num_envs,
            device=device,
            continuous_actions=True,
            max_steps=max_steps,
            n_scouts=n_scouts,
            n_interceptors=n_interceptors,
            n_intruders=n_intruders,
            n_zones=n_zones,
            **kwargs,
        )

        self.obs_dim = get_obs_dim(n_scouts, n_interceptors, n_intruders, n_zones)

        self.defender_indices = []
        self.intruder_indices = []
        for i, agent in enumerate(self.env.agents):
            if hasattr(agent, "agent_type"):
                if agent.agent_type in (SCOUT, INTERCEPTOR):
                    self.defender_indices.append(i)
                elif agent.agent_type == INTRUDER:
                    self.intruder_indices.append(i)

        assert len(self.defender_indices) == self.n_defenders

        self.agent_types = []
        for idx in self.defender_indices:
            self.agent_types.append(self.env.agents[idx].type_id)
        self.agent_types = torch.tensor(self.agent_types, device=device)

    def reset(self):
        """Returns obs (num_envs, n_def, obs_dim), positions (num_envs, n_def, 2)"""
        all_obs = self.env.reset()
        defender_obs = torch.stack([all_obs[i] for i in self.defender_indices], dim=1)
        positions = defender_obs[:, :, 2:4].clone()
        return defender_obs, positions

    def step(self, defender_actions: torch.Tensor):
        """
        Args: defender_actions (num_envs, n_defenders, 2)
        Returns: obs, rewards, dones, info, positions
        """
        all_actions = []
        for i in range(len(self.env.agents)):
            if i in self.defender_indices:
                local_idx = self.defender_indices.index(i)
                all_actions.append(defender_actions[:, local_idx])
            else:
                all_actions.append(torch.zeros(self.num_envs, 2, device=self.device))

        all_obs, all_rewards, dones, all_infos = self.env.step(all_actions)

        defender_obs = torch.stack([all_obs[i] for i in self.defender_indices], dim=1)
        defender_rewards = torch.stack([all_rewards[i] for i in self.defender_indices], dim=1)
        positions = defender_obs[:, :, 2:4].clone()

        info = {}
        if len(all_infos) > 0 and self.defender_indices:
            first_info = all_infos[self.defender_indices[0]]
            if isinstance(first_info, dict):
                info = first_info

        return defender_obs, defender_rewards, dones, info, positions

    def build_adj(self, positions: torch.Tensor, r_comm: float) -> torch.Tensor:
        """Build row-normalized adj matrix. positions: (num_envs, n_def, 2)"""
        diff = positions.unsqueeze(2) - positions.unsqueeze(1)
        dist = torch.linalg.vector_norm(diff, dim=-1)
        adj = (dist <= r_comm).float()
        deg = adj.sum(dim=-1, keepdim=True).clamp(min=1)
        adj = adj / deg
        return adj
    def reset_env(self):
      """Full manual reset — returns obs (num_envs, n_def, obs_dim), positions."""
      all_obs = self.env.reset()
      defender_obs = torch.stack([all_obs[i] for i in self.defender_indices], dim=1)
      positions = defender_obs[:, :, 2:4].clone()
      return defender_obs, positions

    @property
    def action_dim(self) -> int:
        return 2

    @property
    def n_agents(self) -> int:
        return self.n_defenders

### Observation Encoder

3 linear layers — encodes observation from obs_dim to F features.

In [35]:
class ObservationEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ObservationEncoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self._init_weights()

    def _init_weights(self):
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.orthogonal_(layer.weight, gain=2**0.5)
            nn.init.constant_(layer.bias, 0.0)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

### Graph Conv Layers

GCN communication layer — feature aggregation from K-hop neighbors via graph shift operator S.

Already handles batched matmul: `torch.matmul(S, Z)` works for both
`(N, N) @ (N, F)` and `(B, N, N) @ (B, N, F)`.

In [36]:
class GraphConv(nn.Module):
    def __init__(self, F_in, G_out, K):
        super(GraphConv, self).__init__()
        self.F = F_in
        self.G = G_out
        self.K = K

        self.weights = nn.ParameterList(
            [nn.Parameter(torch.empty(F_in, G_out)) for _ in range(K)]
        )
        for p in self.weights:
            nn.init.xavier_uniform_(p)

    def forward(self, X, S):
        Z = X
        accum = X.new_zeros(*X.shape[:-1], self.G)
        for k in range(self.K):
            accum += torch.matmul(Z, self.weights[k])
            Z = torch.matmul(S, Z)
        return accum

### Action Head

2 linear layers — takes aggregated features and outputs action parameters (mean for Gaussian policy).

In [37]:
class ActionHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ActionHead, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.orthogonal_(self.fc2.weight, gain=0.01)
        nn.init.constant_(self.fc2.bias, 0.0)

    def forward(self, G):
        G = F.relu(self.fc1(G))
        G = self.fc2(G)
        return G

### Comm Policy (Continuous Actions)

ObsEncoder → GraphConv → MeanHead → Normal distribution.

**Change from previous:** Uses `Normal` distribution with learnable `log_std`
instead of `Categorical`. Removed dead `self.actionHead`.

In [38]:
class CommPolicy(nn.Module):
    def __init__(self, obs_dim, hidden_dim, action_dim, F_feat, G_feat, K):
        super(CommPolicy, self).__init__()
        self.obsEncoder = ObservationEncoder(obs_dim, hidden_dim, F_feat)
        self.graphConv = GraphConv(F_feat, G_feat, K)
        self.mean_head = ActionHead(G_feat, hidden_dim, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))  # learnable exploration

    def forward(self, obs, S):
        device = next(self.parameters()).device
        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=device)
        else:
            obs = obs.to(device=device, dtype=torch.float32)
        if not torch.is_tensor(S):
            S = torch.as_tensor(S, dtype=torch.float32, device=device)
        else:
            S = S.to(device=device, dtype=torch.float32)

        obs_encode = self.obsEncoder(obs)
        agg_feats = self.graphConv(obs_encode, S)
        mean = self.mean_head(agg_feats)
        return mean

    def get_actions(self, obs, S):
        mean = self.forward(obs, S)
        std = self.log_std.exp().expand_as(mean)
        dist = Normal(mean, std)
        raw_action = dist.sample()
        action = torch.tanh(raw_action)

        log_prob = dist.log_prob(raw_action) - torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1)
        return action, log_prob, entropy

    def evaluate_actions(self, obs, S, actions):
        mean = self.forward(obs, S)
        std = self.log_std.exp().expand_as(mean)
        dist = Normal(mean, std)

        raw_actions = torch.atanh(actions.clamp(-0.999, 0.999))

        log_prob = dist.log_prob(raw_actions) - torch.log(1 - actions.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1)

        entropy = dist.entropy().sum(dim=-1)
        return log_prob, entropy, mean

### Centralized Critic

Shared critic for CTDE — takes per-agent obs, outputs per-agent value estimate.

In [39]:
class CriticNetwork(nn.Module):
    def __init__(self, obs_dim, hidden_dim, device=None):
        super(CriticNetwork, self).__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
        self.device = device
        self._init_weights()
        self.to(self.device)

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc2.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc3.weight, gain=1.0)
        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.constant_(self.fc2.bias, 0.0)
        nn.init.constant_(self.fc3.bias, 0.0)

    def forward(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)
        x = F.relu(self.fc1(obs))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

### Rollout Buffer

**Fix applied:** `torch.long` → `torch.float32` for continuous actions in `get_batches`.

In [40]:
class GNNRolloutBuffer:
    def __init__(self, gamma, gae_lambda, device):
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.adj = []
        self.advantages = None
        self.returns = None
        self.device = device

    def add_timestep(self, obs, actions, rewards, dones, log_probs, values, A):
        self.obs.append(obs)
        self.actions.append(actions)
        self.rewards.append(rewards)
        self.dones.append(dones)
        self.log_probs.append(log_probs)
        self.values.append(values)
        self.adj.append(A)

    def compute_advantages(self, last_values):
        N = self.obs[0].shape[0]  # n_agents
        buffer_size = len(self.obs)
        rewards = torch.stack(self.rewards).to(device=self.device, dtype=torch.float32)
        values = torch.stack(self.values).to(device=self.device, dtype=torch.float32)
        dones = torch.stack(self.dones).to(device=self.device, dtype=torch.float32)

        advantages = torch.zeros((buffer_size, N), dtype=torch.float32, device=self.device)
        last_gae = torch.zeros(N, dtype=torch.float32, device=self.device)

        for t in reversed(range(buffer_size)):
            if t == buffer_size - 1:
                if not torch.is_tensor(last_values):
                    next_value = torch.as_tensor(last_values, dtype=torch.float32, device=self.device)
                else:
                    next_value = last_values.to(device=self.device, dtype=torch.float32)
            else:
                next_value = values[t + 1]

            deltas = rewards[t] + self.gamma * (1 - dones[t]) * next_value - values[t]
            advantages[t] = deltas + self.gamma * self.gae_lambda * (1 - dones[t]) * last_gae
            last_gae = advantages[t]

        returns = advantages + values
        self.advantages = advantages
        self.returns = returns

    def get_batches(self, B):
        perm = torch.randperm(len(self.obs), device=self.device)
        batches = perm.split(B)

        obs = torch.stack(self.obs).to(device=self.device, dtype=torch.float32)
        actions = torch.stack(self.actions).to(device=self.device, dtype=torch.float32)  # [FIX: was torch.long]
        log_probs = torch.stack(self.log_probs).to(device=self.device, dtype=torch.float32)
        adj = torch.stack(self.adj).to(device=self.device, dtype=torch.float32)

        adv_mean = self.advantages.mean()
        adv_std = self.advantages.std() + 1e-8
        advantages = (self.advantages - adv_mean) / adv_std

        for idx in batches:
            yield (
                obs[idx], actions[idx], log_probs[idx],
                advantages[idx], self.returns[idx], adj[idx],
            )

    def clear(self):
        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.adj = []
        self.advantages = None
        self.returns = None

### GNN Trainer (Rewritten for VMAS)

**Major changes:**
- Takes `GuardedTerritoryAdapter` instead of PettingZoo env + GraphBuilder
- No dict↔tensor conversion — adapter returns tensors directly
- Uses `num_envs=1` with squeeze/unsqueeze to match existing buffer shapes
- VMAS auto-resets done environments; we just detect done for metrics

In [ ]:
class GNNTrainer:
    def __init__(
        self,
        adapter: GuardedTerritoryAdapter,
        hidden_dim,
        F_feat,
        G_feat,
        K,
        lr,
        gamma,
        gae_lambda,
        clip_eps,
        value_coef,
        entropy_coef,
        device,
        r_comm,
    ):
        self.device = device
        self.adapter = adapter
        self.num_agents = adapter.n_defenders
        self.r_comm = r_comm

        self.clip_eps = clip_eps
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef

        obs_dim = adapter.obs_dim
        action_dim = adapter.action_dim

        self.comm_policy = CommPolicy(
            obs_dim=obs_dim, hidden_dim=hidden_dim, action_dim=action_dim,
            F_feat=F_feat, G_feat=G_feat, K=K,
        ).to(self.device)
        self.comm_optim = Adam(self.comm_policy.parameters(), lr=lr)

        self.critic = CriticNetwork(obs_dim=obs_dim, hidden_dim=hidden_dim, device=self.device)
        self.critic_optim = Adam(self.critic.parameters(), lr=lr*3)

        self.buffer = GNNRolloutBuffer(gamma=gamma, gae_lambda=gae_lambda, device=self.device)

        self._running_episode_return = 0.0
        self.metrics_history = {
            "policy_loss": [],
            "value_loss": [],
            "entropy": [],
            "mean_bellman_error": [],
            "mean_episode_return": [],
            "mean_episode_rewards": [],
            "scout_reward": [],
            "interceptor_reward": []
        }

        # Initial reset — squeeze out num_envs=1 dim to get (N, obs_dim)
        obs_batched, pos_batched = self.adapter.reset()
        self.current_obs = obs_batched.squeeze(0)       # (N, obs_dim)
        self.current_positions = pos_batched.squeeze(0)  # (N, 2)

    def _safe_mean(self, values):
        if not values:
            return 0.0
        return float(sum(values) / len(values))

    def collect_rollouts(self, num_steps):
        obs_tensor = self.current_obs           # (N, obs_dim)
        positions = self.current_positions       # (N, 2)
        step_mean_rewards = []
        completed_episode_returns = []
        step_mean_scout_rew = []
        step_mean_inter_rew = []

        for _ in range(num_steps):
            # Build adj from current positions: need (1, N, 2) for adapter
            S = self.adapter.build_adj(
                positions.unsqueeze(0), r_comm=self.r_comm
            ).squeeze(0)  # (N, N)

            # Policy forward: (N, obs_dim), (N, N) → actions (N, 2)
            actions, log_probs, entropy = self.comm_policy.get_actions(
                obs=obs_tensor, S=S
            )

            # Critic: (N, obs_dim) → (N,)
            values = self.critic(obs_tensor).detach().squeeze(-1)

            # Step env: adapter expects (num_envs, N, 2)
            act_for_env = actions.unsqueeze(0)  # (1, N, 2)
            next_obs_b, rewards_b, dones_b, info, next_pos_b = self.adapter.step(act_for_env)

            # Squeeze batch dim back out
            next_obs = next_obs_b.squeeze(0)         # (N, obs_dim)
            rewards_tensor = rewards_b.squeeze(0)     # (N,)
            next_positions = next_pos_b.squeeze(0)    # (N, 2)
            done_flag = dones_b.squeeze(0).item()     # scalar bool

            n_s = self.adapter.n_scouts
            scout_rew = rewards_tensor[:n_s].mean().item()
            interceptor_rew = rewards_tensor[n_s:].mean().item()
            if done_flag:
                all_tagged = info.get("n_tagged", torch.tensor(0.0)).item() >= self.adapter.n_intruders
                all_breached = info.get("n_breached", torch.tensor(0.0)).item() >= self.adapter.n_zones
                true_done = all_tagged or all_breached
            else:
                true_done = False
            # Broadcast global done to all agents
            dones_for_gae = torch.full(
                (self.num_agents,), float(true_done),
                dtype=torch.float32, device=self.device,
            )

            # Store in buffer
            self.buffer.add_timestep(
                obs=obs_tensor.detach(),
                actions=actions.detach(),
                rewards=rewards_tensor,
                dones=dones_for_gae,
                log_probs=log_probs.detach(),
                values=values,
                A=S.detach(),
            )

            # Metrics tracking
            step_mean_rewards.append(rewards_tensor.mean().item())
            step_mean_scout_rew.append(scout_rew)
            step_mean_inter_rew.append(interceptor_rew)
            self._running_episode_return += rewards_tensor.mean().item()

            if done_flag:
                completed_episode_returns.append(self._running_episode_return)
                self._running_episode_return = 0.0
                # VMAS auto-resets, so next_obs is already the new episode's obs
                next_obs_b, next_pos_b = self.adapter.reset_env()
                next_obs = next_obs_b.squeeze(0)
                next_positions = next_pos_b.squeeze(0)

            # Advance
            obs_tensor = next_obs
            positions = next_positions

        self.current_obs = obs_tensor
        self.current_positions = positions

        rollout_metrics = {
            "mean_episode_return": self._safe_mean(completed_episode_returns)
            if completed_episode_returns
            else float(self._running_episode_return),
            "mean_episode_rewards": self._safe_mean(step_mean_rewards),
            "mean_scout_rewards": self._safe_mean(step_mean_scout_rew),
            "mean_inter_rewards": self._safe_mean(step_mean_inter_rew)
        }
        self.metrics_history["mean_episode_return"].append(rollout_metrics["mean_episode_return"])
        self.metrics_history["mean_episode_rewards"].append(rollout_metrics["mean_episode_rewards"])
        self.metrics_history["scout_reward"].append(rollout_metrics["mean_scout_rewards"])
        self.metrics_history["interceptor_reward"].append(rollout_metrics["mean_inter_rewards"])
        return obs_tensor, rollout_metrics

    def update(self, last_obs, num_actor_epochs=15, num_critic_epochs=10, B=64):
        with torch.no_grad():
            last_obs_tensor = last_obs.to(device=self.device, dtype=torch.float32)
            last_values = self.critic(last_obs_tensor).squeeze(-1)
        self.buffer.compute_advantages(last_values=last_values)

        policy_losses = []
        entropies = []
        value_losses = []
        bellman_errors = []

        #  actor update 
        for _ in range(num_actor_epochs):
            for (obs, actions, old_log_probs, advantages, returns, A) in self.buffer.get_batches(B):
                new_lp, entropy, _ = self.comm_policy.evaluate_actions(obs, A, actions)

                ratio = torch.exp(new_lp - old_log_probs)
                surr1 = ratio * advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * advantages

                policy_loss = -torch.min(surr1, surr2).mean()
                entropy_loss = entropy.mean()
                loss = policy_loss - self.entropy_coef * entropy_loss

                self.comm_optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.comm_policy.parameters(), max_norm=0.5)
                self.comm_optim.step()

                policy_losses.append(policy_loss.item())
                entropies.append(entropy_loss.item())

        # critic update (fewer epochs) 
        for _ in range(num_critic_epochs):
            for (obs, actions, old_log_probs, advantages, returns, A) in self.buffer.get_batches(B):
                B_size, N, obs_d = obs.shape
                flat_obs = obs.reshape(B_size * N, obs_d)
                flat_returns = returns.reshape(B_size * N)

                pred_values = self.critic(flat_obs).squeeze(-1)
                td_error = flat_returns - pred_values
                value_loss = F.mse_loss(pred_values, flat_returns)
                mean_bellman_error = td_error.abs().mean()

                self.critic_optim.zero_grad()
                value_loss.backward()
                nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=0.5)
                self.critic_optim.step()

                value_losses.append(value_loss.item())
                bellman_errors.append(mean_bellman_error.item())

        self.buffer.clear()
        update_metrics = {
            "policy_loss": self._safe_mean(policy_losses),
            "value_loss": self._safe_mean(value_losses),
            "entropy": self._safe_mean(entropies),
            "mean_bellman_error": self._safe_mean(bellman_errors),
        }
        self.metrics_history["policy_loss"].append(update_metrics["policy_loss"])
        self.metrics_history["value_loss"].append(update_metrics["value_loss"])
        self.metrics_history["entropy"].append(update_metrics["entropy"])
        self.metrics_history["mean_bellman_error"].append(update_metrics["mean_bellman_error"])

        return update_metrics


### Experiment Sweep: K and r_comm

**Changes:**
- `action_dim = 2` (continuous 2D force)
- Uses `GuardedTerritoryAdapter` with `num_envs=1`
- Removed `GraphBuilder` — adapter handles adjacency
- Trainer constructor updated to match new signature

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────
F_dim = 64
G_dim = 64
hidden_dim = 64

lr = 3e-4
gamma = 0.99
gae_lambda = 0.85
clip_eps = 0.2
value_coef = 0.5
entropy_coef = 0.01

TOTAL_TIMESTEPS = 200_000
ROLLOUT_LENGTH = 2048
BATCH_SIZE = 64
NUM_EPOCHS = 10

K_VALUES = [2, 3, 5]
R_COMM_VALUES = [1.0, 1.5, 2.0]
SEED = 42
LOG_EVERY = 5

# ── Environment config ────────────────────────────────────────
NUM_ENVS = 1       # single env; increase for speed later
MAX_STEPS = 200
N_SCOUTS = 3
N_INTERCEPTORS = 3
N_INTRUDERS = 3
N_ZONES = 2
WORLD_SIZE = 2.0

# ── Agent Congfig ──────────
SCOUT_FOV = 0.9
INTERCEPTOR_FOV = 0.6
INTRUDER_SPEED = 0.3
DEFENDER_SPEED = 0.8
TAG_RADIUS = .15

OUTPUT_DIR = "outputs/experiments_k_r"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Sweep: K={K_VALUES}, r_comm={R_COMM_VALUES}")
print(f"TOTAL_TIMESTEPS={TOTAL_TIMESTEPS}, ROLLOUT_LENGTH={ROLLOUT_LENGTH}")

# ── Plotting helper ───────────────────────────────────────────
metrics_to_plot = [
    "policy_loss", "value_loss", "entropy",
    "mean_bellman_error", "mean_episode_return", "mean_episode_rewards",
    "scout_reward", "interceptor_reward"
]

def plot_metrics(metrics_history, title, save_path):
    fig, axes = plt.subplots(4, 2, figsize=(14, 12))
    axes = axes.flatten()
    for i, name in enumerate(metrics_to_plot):
        vals = metrics_history.get(name, [])
        axes[i].plot(range(1, len(vals) + 1), vals, linewidth=1.8)
        axes[i].set_title(name)
        axes[i].set_xlabel("Iteration")
        axes[i].grid(True, alpha=0.3)
    fig.suptitle(title)
    plt.tight_layout()
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.close(fig)

# ── Sweep ─────────────────────────────────────────────────────
results = {}
summary_rows = []

for K_hops in K_VALUES:
    for R_COMM in R_COMM_VALUES:
        config_name = f"K={K_hops}, r_comm={R_COMM}"
        print(f"\n=== Running {config_name} ===")

        torch.manual_seed(SEED)
        np.random.seed(SEED)

        adapter = GuardedTerritoryAdapter(
            num_envs=NUM_ENVS,
            device=device,
            n_scouts=N_SCOUTS,
            n_interceptors=N_INTERCEPTORS,
            n_intruders=N_INTRUDERS,
            n_zones=N_ZONES,
            max_steps=MAX_STEPS,
            world_size=WORLD_SIZE,
            scout_fov=SCOUT_FOV,
            interceptor_fov=INTERCEPTOR_FOV,
            intruder_speed=INTRUDER_SPEED,
            defender_speed=DEFENDER_SPEED,
            tag_radius=TAG_RADIUS,
            intruder_skill=0.0
        )

        trainer = GNNTrainer(
            adapter=adapter,
            hidden_dim=hidden_dim,
            F_feat=F_dim,
            G_feat=G_dim,
            K=K_hops,
            lr=lr,
            gamma=gamma,
            gae_lambda=gae_lambda,
            clip_eps=clip_eps,
            value_coef=value_coef,
            entropy_coef=entropy_coef,
            device=device,
            r_comm=R_COMM,
        )

        steps_done = 0
        iteration = 0

        while steps_done < TOTAL_TIMESTEPS:
            rollout_steps = min(ROLLOUT_LENGTH, TOTAL_TIMESTEPS - steps_done)
            last_obs, rollout_metrics = trainer.collect_rollouts(num_steps=rollout_steps)
            update_metrics = trainer.update(last_obs, num_actor_epochs=NUM_EPOCHS, num_critic_epochs=NUM_EPOCHS//2, B=BATCH_SIZE)

            steps_done += rollout_steps
            iteration += 1

            if iteration == 1 or iteration % LOG_EVERY == 0 or steps_done >= TOTAL_TIMESTEPS:
                print(
                    f"[{config_name}] Iter {iteration}/{TOTAL_TIMESTEPS//iteration} | "
                    f"steps={steps_done}/{TOTAL_TIMESTEPS} | "
                    f"pi_loss={update_metrics['policy_loss']:.4f} | "
                    f"v_loss={update_metrics['value_loss']:.4f} | "
                    f"ent={update_metrics['entropy']:.4f} | "
                    f"bellman={update_metrics['mean_bellman_error']:.4f} | "
                    f"ep_ret={rollout_metrics['mean_episode_return']:.4f} | "
                    f"ep_rew={rollout_metrics['mean_episode_rewards']:.4f} |"
                    f"ep_scout_rew={rollout_metrics['mean_scout_rewards']:4f} |"
                    f"ep_inter_rew: {rollout_metrics["mean_inter_rewards"]:.4f}"
                )
            progress = steps_done / TOTAL_TIMESTEPS
            new_skill = min(1.0, progress * 1.5)  # reaches full skill at ~67% of training
            trainer.adapter.env.scenario.intruder_skill = new_skill

        results[(K_hops, R_COMM)] = trainer.metrics_history

        final_return = trainer.metrics_history["mean_episode_return"][-1]
        final_reward = trainer.metrics_history["mean_episode_rewards"][-1]
        summary_rows.append((K_hops, R_COMM, final_return, final_reward))

        plot_path = os.path.join(OUTPUT_DIR, f"metrics_K{K_hops}_r{R_COMM:.1f}.png")
        plot_metrics(
            trainer.metrics_history,
            title=f"Training Metrics ({config_name})",
            save_path=plot_path,
        )
        print(f"Saved: {plot_path}")

        # Clean up for next config
        del trainer, adapter
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

Using device: cuda
Sweep: K=[2, 3, 5], r_comm=[1.0, 1.5, 2.0]
TOTAL_TIMESTEPS=200000, ROLLOUT_LENGTH=2048

=== Running K=2, r_comm=1.0 ===
[K=2, r_comm=1.0] Iter 1/200000 | steps=2048/200000 | pi_loss=-0.0030 | v_loss=4.7644 | ent=2.8469 | bellman=1.6172 | ep_ret=-12.6414 | ep_rew=-0.0632 |ep_scout_rew=0.454378 |ep_inter_rew: -0.5809
[K=2, r_comm=1.0] Iter 5/40000 | steps=10240/200000 | pi_loss=-0.0041 | v_loss=2.7453 | ent=2.8070 | bellman=1.1956 | ep_ret=-12.7888 | ep_rew=-0.0582 |ep_scout_rew=0.488656 |ep_inter_rew: -0.6050
[K=2, r_comm=1.0] Iter 10/20000 | steps=20480/200000 | pi_loss=-0.0064 | v_loss=4.2232 | ent=2.7420 | bellman=1.5053 | ep_ret=5.5338 | ep_rew=0.0250 |ep_scout_rew=0.646436 |ep_inter_rew: -0.5964
[K=2, r_comm=1.0] Iter 15/13333 | steps=30720/200000 | pi_loss=-0.0094 | v_loss=3.2758 | ent=2.6432 | bellman=1.2697 | ep_ret=13.9724 | ep_rew=0.0869 |ep_scout_rew=0.823958 |ep_inter_rew: -0.6501
[K=2, r_comm=1.0] Iter 20/10000 | steps=40960/200000 | pi_loss=-0.0103 | v_l

### Summary Heatmap

In [ ]:
return_matrix = np.full((len(K_VALUES), len(R_COMM_VALUES)), np.nan, dtype=np.float32)
for K_hops, R_COMM, final_return, _ in summary_rows:
    i = K_VALUES.index(K_hops)
    j = R_COMM_VALUES.index(R_COMM)
    return_matrix[i, j] = final_return

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(return_matrix, cmap="viridis")
ax.set_xticks(range(len(R_COMM_VALUES)))
ax.set_yticks(range(len(K_VALUES)))
ax.set_xticklabels([str(v) for v in R_COMM_VALUES])
ax.set_yticklabels([str(v) for v in K_VALUES])
ax.set_xlabel("r_comm")
ax.set_ylabel("K")
ax.set_title("Final Mean Episode Return")

for i in range(len(K_VALUES)):
    for j in range(len(R_COMM_VALUES)):
        ax.text(j, i, f"{return_matrix[i, j]:.2f}", ha="center", va="center", color="white")

fig.colorbar(im, ax=ax)
plt.tight_layout()
summary_path = os.path.join(OUTPUT_DIR, "summary_final_return_heatmap.png")
plt.savefig(summary_path, dpi=180)
plt.show()
print(f"Saved summary heatmap: {summary_path}")

print("\nFinal summary (K, r_comm, final_return, final_reward):")
for row in summary_rows:
    print(row)